# Análisis de ventas por empleado | Northwind

Este notebook presenta un análisis de las ventas registradas en
la base de datos Northwind, combinando SQL, Python, pandas y Plotly
para consultar, procesar y visualizar información.

## Objetivos

- Obtener la cantidad de pedidos gestionados por empleado.
- Calcular las unidades vendidas y el importe estimado de sus ventas.
- Analizar los promedios de unidades vendidas.
- Identificar empleados por encima y por debajo del promedio.
- Visualizar los resultados mediante gráficos interactivos.

## Metodología

1. **SQLite:** conexión y extracción de datos mediante subconsultas SQL.
2. **pandas:** organización, filtrado y análisis estadístico.
3. **Plotly:** visualización de unidades vendidas e importes estimados.

> **Nota:** los importes se calculan utilizando los precios de la
> tabla `Products`. Por ello, son estimaciones y no representan
> necesariamente los ingresos históricos ni las ganancias reales.
---

In [7]:
from pathlib import Path    # para localizar el archivo .db o base de datos
import sqlite3              # para conectarnos con la bd
import pandas as pd         # tablas y datos

directorio_actual = Path.cwd()      # current working directory

print(f"current working directory: {directorio_actual}")

# Posibles ubicaciones de Northwind.db (Aunque ya se donde esta)
rutas = [
    directorio_actual / "Northwind.db",
    directorio_actual.parent / "Northwind.db",
    directorio_actual.parent / "data/Northwind.db",
    directorio_actual / "SQL-desde-0/northwind/data/Northwind.db"
]

# Identifica la ruta correcta
ruta_db = next((ruta for ruta in rutas if ruta.is_file()), None)

# Escepcion para que nos avise por si no encuentra la ruta
if ruta_db is None:
    raise FileNotFoundError("No se encontro la bd Northwind.db")

# Abrir la base de datos en modo solo lectura
conexion = sqlite3.connect(
    f"{ruta_db.resolve().as_uri()}?mode=ro", # Elegimos ro read only porque solo vamos a obtener datos, no vamos a escribir en este análisis.
    uri=True
)

print(f"Conectado a: {ruta_db.resolve()}")

current working directory: /home/ssail/GitHubDir/CoursesDaltoDevLab/SQL-desde-0/northwind/notebooks
Conectado a: /home/ssail/GitHubDir/CoursesDaltoDevLab/SQL-desde-0/northwind/data/Northwind.db


In [8]:
consulta_ventas = """
SELECT
    e.EmployeeID,
    e.FirstName,
    e.LastName,
    --- Subconsulta pedidos por empleado
    (
        SELECT COUNT(o.OrderID)
        FROM Orders o
        WHERE e.EmployeeID = o.EmployeeID
    ) as Pedidos,
    COALESCE(
        (
            SELECT
                sum(Quantity)
            FROM Orders o, OrderDetails od
            WHERE e.EmployeeID = o.EmployeeID
                AND
                o.OrderID = od.OrderID
        ),
        0
    ) as Unidades_Vendidas,
    COALESCE(
        (
            SELECT sum(p.Price * od.Quantity)
            FROM Products p, Orders o, OrderDetails od
            WHERE e.EmployeeID = o.EmployeeID
            AND o.OrderID = od.OrderID
            AND od.ProductID = p.ProductID
        ),
        0    
    ) AS Importe_estimado
FROM
    Employees e
"""

df_ventas = pd.read_sql_query(consulta_ventas, conexion)

display(df_ventas)

,EmployeeID,FirstName,LastName,Pedidos,Unidades_Vendidas,Importe_estimado
0,1,Nancy,Davolio,29,1924,57690.39
1,2,Andrew,Fuller,20,1315,32503.16
2,3,Janet,Leverling,31,1725,42838.35
3,4,Margaret,Peacock,40,3232,105696.50
4,5,Steven,Buchanan,11,778,27480.80
5,6,Michael,Suyama,18,1094,25399.25
6,7,Robert,King,14,733,39772.30
7,8,Laura,Callahan,27,1293,39309.38
8,9,Anne,Dodsworth,6,649,15734.10
9,10,Adam,West,0,0,0.00


In [9]:
# Promedio de todos los empleados, incluyendo aquellos con 0 ventas
promedio_general = df_ventas["Unidades_Vendidas"].mean()

# Seleccionamos únicamente a los empleados que sí vendieron
df_activos = df_ventas[
    df_ventas["Unidades_Vendidas"] > 0
].copy()

# Promedio de los empleados con ventas
promedio_activos = df_activos["Unidades_Vendidas"].mean()

print(f"Promedio general: {promedio_general:.2f}")
print(f"Promedio de empleados con ventas: {promedio_activos:.2f}")

Promedio general: 1274.30
Promedio de empleados con ventas: 1415.89


In [10]:
# Empleados por encima del promedio
df_superiores = df_ventas[
    df_ventas["Unidades_Vendidas"] > promedio_activos
].sort_values(
    by="Importe_estimado",
    ascending=False
)

# Empleados por debajo del promedio
df_inferiores = df_ventas[
    df_ventas["Unidades_Vendidas"] < promedio_activos
].sort_values(
    by="Importe_estimado",
    ascending=False
)

print("EMPLEADOS POR ENCIMA DEL PROMEDIO")
display(df_superiores)

print("EMPLEADOS POR DEBAJO DEL PROMEDIO")
display(df_inferiores)

EMPLEADOS POR ENCIMA DEL PROMEDIO


,EmployeeID,FirstName,LastName,Pedidos,Unidades_Vendidas,Importe_estimado
3,4,Margaret,Peacock,40,3232,105696.50
0,1,Nancy,Davolio,29,1924,57690.39
2,3,Janet,Leverling,31,1725,42838.35


EMPLEADOS POR DEBAJO DEL PROMEDIO


,EmployeeID,FirstName,LastName,Pedidos,Unidades_Vendidas,Importe_estimado
6,7,Robert,King,14,733,39772.30
7,8,Laura,Callahan,27,1293,39309.38
1,2,Andrew,Fuller,20,1315,32503.16
4,5,Steven,Buchanan,11,778,27480.80
5,6,Michael,Suyama,18,1094,25399.25
8,9,Anne,Dodsworth,6,649,15734.10
9,10,Adam,West,0,0,0.00


In [11]:
import plotly.express as px

df_ventas_desc_unidades = df_ventas.sort_values(
    by="Unidades_Vendidas",
    ascending=False
)

fig_Unidades_Vendidas = px.bar(
    df_ventas_desc_unidades,
    x="FirstName",
    y="Unidades_Vendidas",
    title="Unidades vendidas por empleado",
    labels={
        "FirstName": "Empleado",
        "Unidades_Vendidas": "Unidades vendidas"
    }
)

fig_Unidades_Vendidas.add_hline(
    y=promedio_activos,
    line_dash="dash",
    annotation_text="Promedio de empleados con ventas"
)

fig_Unidades_Vendidas.add_hline(
    y=promedio_general,
    line_dash="dash",
    annotation_text="Promedio de empleados en general"
)

fig_Unidades_Vendidas.show()

In [12]:
df_ventas_desc_importe = df_ventas.sort_values(
    by="Importe_estimado",
    ascending=False
)

fig_importe_estimado = px.bar(
    df_ventas_desc_importe,
    x="FirstName",
    y="Importe_estimado",
    title="Importe_Estimado",
    labels={
        "FirstName": "Empleados",
        "Importe_estimado": "Importe Estimado"
    }
)

fig_importe_estimado.show()

---

In [20]:
raiz_repo = ruta_db.resolve().parents[3]

print(raiz_repo)

directorio_publicacion = raiz_repo / "docs" / "sql" / "db_northwind" / "subconsultas_ventas_por_empleado"
directorio_publicacion.mkdir(
    parents=True,
    exist_ok=True
)

print(directorio_publicacion)

/home/ssail/GitHubDir/CoursesDaltoDevLab
/home/ssail/GitHubDir/CoursesDaltoDevLab/docs/sql/db_northwind/subconsultas_ventas_por_empleado


In [21]:
# Exportar gráfico de unidades
fig_Unidades_Vendidas.write_html(
    directorio_publicacion / "unidades_vendidas.html",
    include_plotlyjs="cdn"
)

# Exportar gráfico de importes
fig_importe_estimado.write_html(
    directorio_publicacion / "importe_estimado.html",
    include_plotlyjs="cdn"
)

print("Gráficas exportadas correctamente.")

Gráficas exportadas correctamente.
